In [1]:
# CẬP NHẬT CELL 1
!pip install -q -U accelerate transformers peft bitsandbytes trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.0 MB/s eta 0:00:00


In [2]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
import matplotlib.pyplot as plt

# Cấu hình hằng số theo project
MODEL_ID = "google/gemma-4-E2B-it" # Cập nhật đúng ID trên HuggingFace
DATA_PATH = "/kaggle/input/datasets/hauuto/vietnamese-elementary-math/data_warehouse.csv" # Sửa lại tên thư mục input nếu cần
OUTPUT_DIR = "/kaggle/working/models/m3_gemma"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final"
LOGS_DIR = f"{OUTPUT_DIR}/logs"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

In [3]:
# ==========================================
# CẬP NHẬT LẠI CELL 3: RANDOM SPLIT & ÉP CÂN DỮ LIỆU
# ==========================================
import pandas as pd
from datasets import Dataset

print(f"Đang đọc dữ liệu từ: {DATA_PATH}")
try:
    df = pd.read_csv(DATA_PATH)
    
    # Bỏ các dòng thiếu question hoặc answer
    df = df.dropna(subset=['question', 'answer'])
    print(f"Tổng số dòng hợp lệ trước khi chia: {len(df)}")
    
except FileNotFoundError:
    print(f"Không tìm thấy {DATA_PATH}.")
    df = pd.DataFrame({"instruction": ["Giải toán"], "question": ["1+1="], "answer": ["2"]})

def format_instruction(row):
    instruction = row.get('instruction')
    
    # Xử lý dứt điểm các instruction bị null, NaN, hoặc chứa rác
    if pd.isna(instruction) or str(instruction).strip() in ['[null]', '', 'main/train', '[]']:
        instruction = 'Hãy giải bài toán sau từng bước:'
        
    question = row['question']
    answer = row['answer']
    return f"<bos><start_of_turn>user\n{instruction}\n{question}<end_of_turn>\n<start_of_turn>model\n{answer}<end_of_turn><eos>"

# Áp dụng định dạng Prompt của Gemma
df["text"] = df.apply(format_instruction, axis=1)

# Convert toàn bộ Pandas DataFrame sang HuggingFace Dataset
full_dataset = Dataset.from_pandas(df[["text"]])

# CHIA TẬP DỮ LIỆU NGẪU NHIÊN (SEED=42 để có thể tái tạo kết quả)
# Bước 1: Lấy ra 90% làm Train, 10% còn lại làm tập tạm (Temp)
split_90_10 = full_dataset.train_test_split(test_size=0.10, seed=42)
train_dataset = split_90_10['train']
temp_dataset = split_90_10['test']

# Bước 2: Chia đôi tập Temp (10%) thành Validation (5%) và Test (5%)
split_5_5 = temp_dataset.train_test_split(test_size=0.50, seed=42)
eval_dataset = split_5_5['train']
test_dataset = split_5_5['test']

# ==========================================
# [MỚI CHÈN VÀO]: CẮT GIẢM TẬP TRAIN XUỐNG 20K MẪU ĐỂ QUA ẢI 9 TIẾNG
# ==========================================
if len(train_dataset) > 20000:
    train_dataset = train_dataset.select(range(20000))

print("-" * 30)
print(f"Số lượng mẫu huấn luyện (Train) ĐÃ RÚT GỌN : {len(train_dataset)}")
print(f"Số lượng mẫu đánh giá (Validation)         : {len(eval_dataset)}")
print(f"Số lượng mẫu kiểm tra (Test)               : {len(test_dataset)}")
print("-" * 30)

# Lưu tập Test ra file (tùy chọn) để dùng cho việc chấm điểm (evaluate) ở các bước sau
test_df = test_dataset.to_pandas()
test_df.to_csv(f"{OUTPUT_DIR}/test_set_random.csv", index=False)

Đang đọc dữ liệu từ: /kaggle/input/datasets/hauuto/vietnamese-elementary-math/data_warehouse.csv
Tổng số dòng hợp lệ trước khi chia: 96502
------------------------------
Số lượng mẫu huấn luyện (Train) ĐÃ RÚT GỌN : 20000
Số lượng mẫu đánh giá (Validation)         : 4825
Số lượng mẫu kiểm tra (Test)               : 4826
------------------------------


In [4]:
# ==========================================
# CELL 4: NẠP MÔ HÌNH VÀ CẤU HÌNH PEFT LORA
# ==========================================
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

print("--- BƯỚC 2: NẠP MÔ HÌNH NỀN VÀ LORA ---")

# Dọn rác bộ nhớ (đề phòng chạy lại nhiều lần)
torch.cuda.empty_cache()
gc.collect()

# Cấu hình lượng tử hóa 4-bit chuẩn cho Gemma 4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Khởi tạo Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("Đang nạp trọng số mô hình... (Hệ thống sẽ tự động cắt đôi qua 2 GPU T4)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto" # Mấu chốt phân bổ VRAM
)

# Giải phóng bộ nhớ đệm (cần thiết cho quá trình huấn luyện)
model.config.use_cache = False 

# Cấu hình LoRA (Chỉ học các trọng số bổ sung)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
print("\nSố lượng tham số Trainable:")
model.print_trainable_parameters()

--- BƯỚC 2: NẠP MÔ HÌNH NỀN VÀ LORA ---


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Đang nạp trọng số mô hình... (Hệ thống sẽ tự động cắt đôi qua 2 GPU T4)


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]


Số lượng tham số Trainable:
trainable params: 37,920,768 || all params: 5,142,218,272 || trainable%: 0.7374


In [5]:
# ==========================================
# CELL 5: TIỀN XỬ LÝ VÀ HUẤN LUYỆN (CỨU CÁNH CHECKPOINTING & LƯU NHÁP)
# ==========================================
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import gc
import torch

print("\n--- BƯỚC 3: CHẶT NGẮN DỮ LIỆU BẢO VỆ VRAM ---")

def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=256 
    )

cols_to_remove = train_dataset.column_names
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=cols_to_remove)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=cols_to_remove)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("\n--- BƯỚC 4: KHỞI ĐỘNG HUẤN LUYỆN ---")

model.enable_input_require_grads()
model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="no",
    logging_steps=10,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    weight_decay=0.01,
    optim="paged_adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    num_train_epochs=1,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    
    # [ĐÃ SỬA]: Tính năng lưu nháp đề phòng Kaggle ngắt mạng
    save_strategy="steps",
    save_steps=200,      # Cứ 200 bước (~1600 mẫu) thì lưu 1 bản
    save_total_limit=2,  # Chỉ giữ 2 bản gần nhất cho đỡ tốn bộ nhớ
    
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False} 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("Đã bật khiên Gradient Checkpointing. Bắt đầu quá trình Train...")
torch.cuda.empty_cache()
gc.collect()

trainer.train()

print("\n--- BƯỚC 5: LƯU MÔ HÌNH HOÀN CHỈNH ---")
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print(f"🎉 CHIẾN THẮNG! Mô hình của bạn đã hoàn thiện tại: {FINAL_MODEL_DIR}")


--- BƯỚC 3: CHẶT NGẮN DỮ LIỆU BẢO VỆ VRAM ---


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4825 [00:00<?, ? examples/s]


--- BƯỚC 4: KHỞI ĐỘNG HUẤN LUYỆN ---
Đã bật khiên Gradient Checkpointing. Bắt đầu quá trình Train...


Step,Training Loss
10,2.811518
20,2.446292
30,1.673828
40,1.176025
50,0.950364
60,0.839612
70,0.810105
80,0.811758
90,0.810966
100,0.771005



--- BƯỚC 5: LƯU MÔ HÌNH HOÀN CHỈNH ---
🎉 CHIẾN THẮNG! Mô hình của bạn đã hoàn thiện tại: /kaggle/working/models/m3_gemma/final


In [6]:
# ==========================================
# CELL 6: ĐÁNH GIÁ, INFERENCE VÀ XUẤT LOG
# ==========================================
import pandas as pd
import torch
import math
import os

print("--- BƯỚC 6: ĐÁNH GIÁ VÀ XUẤT KẾT QUẢ ---")

# 1. CHUẨN BỊ TẬP TEST
# Cắt ngắn tập test xuống 256 tokens giống hệt lúc train để tránh OOM
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=test_dataset.column_names)

# Đưa mô hình về chế độ suy luận, dọn rác VRAM
model.eval()
model.config.use_cache = True # Bật lại cache để sinh chữ nhanh hơn
torch.cuda.empty_cache()

# 2. ĐÁNH GIÁ TỔNG QUAN (EVALUATION)
print("\nĐang tính toán Loss và Perplexity trên tập Test...")
test_results = trainer.evaluate(eval_dataset=tokenized_test)

# Tính Perplexity (Chỉ số độ bối rối của ngôn ngữ - càng thấp càng tốt)
try:
    test_results["perplexity"] = math.exp(test_results["eval_loss"])
except OverflowError:
    test_results["perplexity"] = float("inf")

print(f"📊 Kết quả Đánh giá Tập Test:")
print(f" - Test Loss: {test_results['eval_loss']:.4f}")
print(f" - Perplexity: {test_results['perplexity']:.4f}")

# Lưu kết quả tổng quan ra CSV
pd.DataFrame([test_results]).to_csv(f"{LOGS_DIR}/test_metrics.csv", index=False)

# 3. TRÍCH XUẤT LỊCH SỬ HUẤN LUYỆN (TRAIN LOGS)
print("\nĐang xuất lịch sử huấn luyện ra CSV...")
log_history = trainer.state.log_history
log_df = pd.DataFrame(log_history)
# Lọc bỏ các dòng không chứa thông tin loss
log_df = log_df.dropna(subset=['loss', 'eval_loss'], how='all')
log_df.to_csv(f"{LOGS_DIR}/training_history.csv", index=False)

# 4. CHẠY THỬ NGHIỆM THỰC TẾ (INFERENCE)
print("\nĐang sinh câu trả lời cho 10 mẫu ngẫu nhiên (Inference)...")
sample_test = test_dataset.shuffle(seed=42).select(range(min(10, len(test_dataset))))

inference_results = []

with torch.no_grad(): # Tắt tính toán đạo hàm để tiết kiệm 100% VRAM
    for i, item in enumerate(sample_test):
        # Tách prompt (câu hỏi) và ground_truth (đáp án chuẩn) từ cột 'text'
        parts = item['text'].split("<start_of_turn>model\n")
        prompt = parts[0] + "<start_of_turn>model\n"
        ground_truth = parts[1].replace("<end_of_turn><eos>", "").strip() if len(parts) > 1 else ""
        
        # Tokenize đầu vào
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        # Sinh văn bản (Giới hạn tối đa 150 tokens mới để chống OOM)
        outputs = model.generate(
            **inputs, 
            max_new_tokens=150,
            temperature=0.7,      # Độ sáng tạo vừa phải cho toán học
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
        
        # Decode kết quả (cắt bỏ phần prompt ban đầu để lấy nguyên đáp án)
        generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        
        # Dọn VRAM ngay sau mỗi vòng lặp
        del inputs, outputs
        torch.cuda.empty_cache()
        
        inference_results.append({
            "Prompt": prompt,
            "Ground_Truth": ground_truth,
            "AI_Prediction": generated_text
        })
        print(f"✅ Đã sinh xong mẫu {i+1}/10")

# 5. XUẤT KẾT QUẢ INFERENCE RA CSV
inference_df = pd.DataFrame(inference_results)
inference_csv_path = f"{OUTPUT_DIR}/inference_sample_results.csv"
inference_df.to_csv(inference_csv_path, index=False)

print("\n🎉 HOÀN TẤT TOÀN BỘ QUÁ TRÌNH!")
print(f"📁 Log huấn luyện lưu tại: {LOGS_DIR}/training_history.csv")
print(f"📁 Chỉ số Test lưu tại: {LOGS_DIR}/test_metrics.csv")
print(f"📁 Kết quả sinh chữ thực tế lưu tại: {inference_csv_path}")

--- BƯỚC 6: ĐÁNH GIÁ VÀ XUẤT KẾT QUẢ ---


Map:   0%|          | 0/4826 [00:00<?, ? examples/s]


Đang tính toán Loss và Perplexity trên tập Test...


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.57 GiB is free. Including non-PyTorch memory, this process has 12.87 GiB memory in use. Of the allocated memory 12.62 GiB is allocated by PyTorch, and 115.35 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)